In [5]:
import os
import sys
from datetime import datetime

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns
import infinite
reload(plotting)
reload(pinns)
reload(infinite)
import numpy as np
import sympy as sp
import pickle
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf,compute_training_errors
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison
from calflops import calculate_flops
set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [6]:
model_u, model_k = build_models_KAN(
    device,
    hidden_units=25,
    grid_size=5,
    spline_order=3,
)

In [7]:
flops, macs, params = calculate_flops(
    model=model_u, 
    input_shape=(1, model_u.layers[0].in_features)
)

print(f"FLOPs totales por inferencia: {flops}")
print(f"Parámetros contados por el perfilador: {params}")


------------------------------------- Calculate Flops Results -------------------------------------
Notations:
number of parameters (Params), number of multiply-accumulate operations(MACs),
number of floating-point operations (FLOPs), floating-point operations per second (FLOPS),
fwd FLOPs (model forward propagation FLOPs), bwd FLOPs (model backward propagation FLOPs),
default model backpropagation takes 2.00 times as much computation as forward propagation.

Total Training Params:                                                  13.25 K 
fwd MACs:                                                               11.93 KMACs
fwd FLOPs:                                                              23.93 KFLOPS
fwd+bwd MACs:                                                           35.77 KMACs
fwd+bwd FLOPs:                                                          71.78 KFLOPS

-------------------------------- Detailed Calculated FLOPs Results --------------------------------
Each module cac

In [4]:
def calculate_kan_complexity(layers_hidden, grid_size=5, spline_order=3, batch_size=1):
    total_params = 0
    total_flops = 0
    
    for din, dout in zip(layers_hidden[:-1], layers_hidden[1:]):
        # 1. Parámetros por capa (efficient-kan)
        params = (din * dout) * (grid_size + spline_order + 2)
        total_params += params
        
        # 2. FLOPs por capa (estimación analítica)
        flops_silu = 4 * din                                    # Activación base SiLU
        flops_base_linear = 2 * din * dout                      # Proyección lineal base
        flops_spline_bases = din * (2 * (grid_size + spline_order)) # Evaluación B-splines
        flops_spline_linear = 2 * din * dout * (grid_size + spline_order) # Proyección lineal del spline
        flops_scaler = din * dout                               # Multiplicación por spline_scaler
        
        layer_flops = batch_size * (flops_silu + flops_base_linear + flops_spline_bases + flops_spline_linear + flops_scaler)
        total_flops += layer_flops
        
    return total_params, total_flops

# Ejemplo de uso:
layers = [2, 25, 25, 25, 1]
params, flops = calculate_kan_complexity(layers, grid_size=5, spline_order=3, batch_size=1)
print(f"Parámetros: {params:,} | FLOPs: {flops:,}")

Parámetros: 13,250 | FLOPs: 26,715
